# Exploring CTA bus ridership

This notebook explores CTA bus ridership since 2001, understanding the available data and the system as a whole. 

After exploration, it also generates datasets used in other notebooks. (Run this notebook (it writes `data/derived/`), before `holidays.ipynb` and `seasonality.ipynb`).

Exploration is a first step before setting up a before/ after comparison for the 10 minute frequent network program,
an analysis which which will be fleshed out in `frequent_network_analysis.ipynb` (currently under construction/ being rebuilt).


### Outline
0. Import packages and Load in the data
    - CTA daily bus ridership dataset: [CTA Ridership – Bus Routes – Daily Totals by Route](https://data.cityofchicago.org/Transportation/CTA-Ridership-Bus-Routes-Daily-Totals-by-Route/jyb9-n7fm/about_data)
    - CTA monthly bus ridership dataset:   [CTA – Ridership – Bus Routes – Monthly Day-Type Averages & Totals](https://data.cityofchicago.org/Transportation/CTA-Ridership-Bus-Routes-Monthly-Day-Type-Averages/bynn-gwxy)
    - CTA route geometry, current and 2015: [CTA - Bus Routes](https://data.cityofchicago.org/Transportation/CTA-Bus-Routes/6uva-a5ei/about_data), [CTA - Bus Routes - KML](https://data.cityofchicago.org/Transportation/CTA-Bus-Routes-KML-Deprecated-February-2015-/atza-xq2n)
1. Data quality checks and cross checks
   - missing data, NANs, etc.
   - cross check datasets
   - define/check weeks and look for endpoint issues
2. Define cooridors: how do we treat routes that cover the same streets?
    - take the same base number, look at the family
    - ignore R routes from 2013 redline project
3. Explore total ridership for the full dataset, 2001-2026
4. Explore ridership by route:
   - the network mapped, coloured by ridership: current geometry and the 2015 snapshot
   - which routes have highest ridership, raw and per mile of route
   - route change over time: routes that have started/ended; route growth or shrinkage relative to system average
   - route recovery since pandemic
5. Explore ridership by day of the week
6. Save the cleaned data for the companion notebooks

Two topics have their own notebooks:

- **`holidays.ipynb`** — which days CTA actually runs a holiday schedule, and what that does to ridership.
- **`seasonality.ipynb`** — the within-year profile, with the trend removed and holiday weeks held out.


### Notes about the data
 Coming soon.

### Open questions/ future improvements
- think more carefully about how express / branch routes (`X49`, `53A`, …) fold into corridors.
- 39 routes have no geometry in either portal file, so they are absent from the map and the
  lifespan panels — mostly `R` shuttles, `X` expresses and special-event IDs.
- confirm against CTA maps that the six `X` expresses given a parent route's length in §4.b
  really do run their parent's full length.
- Before/after window lengths for the event study.
- The definition of "usual" used in `holidays.ipynb` — flagged inline there.

## 0. Setup and load the data

### 0.a) Import necessary packages and define general plotting parameters

In [ ]:
import re
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.collections import LineCollection
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter
from shapely.geometry import LineString, MultiLineString

# Okabe-Ito: the published colour-vision-deficiency-safe categorical set.
BLUE, ORANGE, GREEN, PURPLE = '#0072B2', '#D55E00', '#009E73', '#CC79A7'
GRAY, INK = '#C9C9C9', '#333333'

# Ridership regimes. These are a COLOUR SCHEME for reading charts over time, not an
# analysis grouping: any analysis that needs a particular window defines it locally and
# prints what it used. 2020 gets its own colour because it is not comparable to anything
# else; the recovery years are lighter shades of it because the system has not returned
# to the pre-2020 level; 2025-present is distinct because the Frequent Network rollout
# begins 2025-03-23, so those years are not a clean baseline for anything.
ERAS = [('pre-2020',     2001, 2019, BLUE),
        ('2020',         2020, 2020, ORANGE),
        ('2021-2022',    2021, 2022, '#EE8A4E'),
        ('2023-2024',    2023, 2024, '#F5BE99'),
        ('2025-present', 2025, 2026, GREEN)]
ERA_ORDER = [e[0] for e in ERAS]
ERA_COLOR = {e[0]: e[3] for e in ERAS}

def era(year):
    """Map a calendar year to its ridership regime."""
    for name, lo, hi, _ in ERAS:
        if lo <= year <= hi:
            return name
    return None

# The 20 Frequent Network routes, as CTA labels them. Defined here because several
# sections need it, including the corridor check in section 2.
FREQ = ['J14', '4', '9', '12', '20', '34', '47', '49', '53', '54',
        '55', '60', '63', '66', '72', '77', '79', '81', '82', '95']
FREQ_SET = set(FREQ)

# Illinois State Plane East, in feet. Bearings and lengths need a planar CRS, so every
# geometry is reprojected to this before anything is measured off it.
CRS = 'EPSG:3435'

plt.rcParams.update({
    'figure.dpi': 110, 'font.size': 9,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': '#9A9A9A', 'axes.grid': True,
    'grid.color': '#E8E8E8', 'grid.linewidth': 0.8,
})
fmt_riders = FuncFormatter(lambda v, _: f'{v*1e-6:.1f}M' if v >= 1e6 else f'{v*1e-3:.0f}k')

### 0.b) Read in the main CTA bus data file

[CTA Ridership – Bus Routes – Daily Totals by Route](https://data.cityofchicago.org/Transportation/CTA-Ridership-Bus-Routes-Daily-Totals-by-Route/jyb9-n7fm/about_data)
  (Chicago Data Portal, dataset `jyb9-n7fm`), pulled via the Socrata API:
  `https://data.cityofchicago.org/resource/jyb9-n7fm.csv`.
  Fields: `route`, `date`, `daytype` (**W** = weekday, **A** = Saturday, **U** =
  Sunday/holiday), `rides`.

```
Re-download if needed:
!curl -s "https://data.cityofchicago.org/resource/jyb9-n7fm.csv?\$limit=2000000&\$order=route,date" -o data/cta_bus_daily.csv
```

In [ ]:
df_raw = pd.read_csv('data/cta_bus_daily.csv', dtype={'route': str, 'daytype': str})
print(f'rows read : {len(df_raw):,}')
print(f'columns   : {list(df_raw.columns)}')
df_raw.head()

### 0.c) Read in *Monthly Averages and Totals File* to check against and get route names

The daily file doesn't have route names, but here is another file that aggregates by month on the same portal, along
with day-type averages we can check our own aggregation against:

[CTA – Ridership – Bus Routes – Monthly Day-Type Averages & Totals](https://data.cityofchicago.org/Transportation/CTA-Ridership-Bus-Routes-Monthly-Day-Type-Averages/bynn-gwxy)
(`bynn-gwxy`), saved to `data/cta_bus_monthly.csv`:

```
curl "https://data.cityofchicago.org/resource/bynn-gwxy.csv?$limit=50000&$order=route,month_beginning" \
     -o data/cta_bus_monthly.csv
```

Columns: `route`, `routename`, `month_beginning`, `avg_weekday_rides`, `avg_saturday_rides`,
`avg_sunday_holiday_rides`, `monthtotal`.

In [ ]:
df_monthly = pd.read_csv('data/cta_bus_monthly.csv', dtype={'route': str},
                         parse_dates=['month_beginning'])
print(f'rows {len(df_monthly):,}   routes {df_monthly.route.nunique()}   '
      f'{df_monthly.month_beginning.min().date()} .. '
      f'{df_monthly.month_beginning.max().date()}')

# Names change over time, so take each route's most recent name and list the changes.
# route_names is the only source of route names in the notebook -- the daily file has none.
route_names = df_monthly.sort_values('month_beginning').groupby('route').routename.last()
routes_renamed = df_monthly.groupby('route').routename.nunique()
routes_renamed = routes_renamed[routes_renamed > 1]
print(f'\nroutes renamed at least once: {len(routes_renamed)}')
for route in routes_renamed.index:
    print(f'  {route:<5} '
          + ' -> '.join(df_monthly.loc[df_monthly.route == route, 'routename'].unique()))

### 0.d) Read in the route geometry files

Two files, because routes change over time and the portal keeps only one historical snapshot:

- **current** — [CTA - Bus Routes](https://data.cityofchicago.org/Transportation/CTA-Bus-Routes/6uva-a5ei/about_data) (`6uva-a5ei`), updated 2025-01-08, 127 routes.
- **2015** — [CTA - Bus Routes - KML](https://data.cityofchicago.org/Transportation/CTA-Bus-Routes-KML-Deprecated-February-2015-/atza-xq2n) (`atza-xq2n`), deprecated 2015-02, 140 routes.

Both are cached under `data/geo/` (gitignored) and re-download if missing. Used in §4.a for the
map and the orientation classes, and in §4.b for route length.

In [ ]:
GEO_DIR = Path('data/geo')
GEO_DIR.mkdir(parents=True, exist_ok=True)
GEO_SRC = {                                  # vintage -> (download url, local cache path)
    'current': ('https://data.cityofchicago.org/resource/6uva-a5ei.geojson?$limit=1000',
                GEO_DIR / 'cta_routes_current.geojson'),
    '2015':    ('https://data.cityofchicago.org/api/views/atza-xq2n/files/'
                '8xsv05KeqbpuBTION_ykEi4pR2dd4rT6aUPLdLbGTAM?filename=CTABusRoutes.kml',
                GEO_DIR / 'cta_routes_2015.kml'),
}
for vintage, (url, path) in GEO_SRC.items():
    if not path.exists():
        print(f'downloading {vintage} ...')
        urlretrieve(url, path)
    print(f'{vintage:8s} {str(path):<34} {path.stat().st_size / 1e6:.1f} MB')

# The current file is GeoJSON, so geopandas reads it directly.
geo_current = gpd.read_file(GEO_SRC['current'][1])[['route', 'geometry']]
geo_current['route'] = geo_current['route'].astype(str)

# The 2015 file is KML. Parse the placemarks with regexes rather than depend on a KML driver
# being installed: one <Placemark> per route, each holding one or more <coordinates> runs.
kml_text = open(GEO_SRC['2015'][1], encoding='utf-8', errors='replace').read()
kml_rows = []
for placemark in re.findall(r'<Placemark>(.*?)</Placemark>', kml_text, re.S):
    name_tag = re.search(r'<name>(.*?)</name>', placemark, re.S)
    coord_blocks = re.findall(r'<coordinates>(.*?)</coordinates>', placemark, re.S)
    if not name_tag or not coord_blocks:
        continue
    # Each block is whitespace-separated "lon,lat,alt" triples; keep lon/lat, drop altitude.
    line_parts = [LineString(points) for points in
                  ([tuple(map(float, t.split(',')[:2])) for t in block.split() if ',' in t]
                   for block in coord_blocks)
                  if len(points) >= 2]
    if line_parts:
        kml_rows.append({'route': name_tag.group(1).strip(),
                         'geometry': MultiLineString(line_parts)})
# dissolve() merges the placemarks that share a route id into one geometry per route.
geo_2015 = gpd.GeoDataFrame(kml_rows, crs='EPSG:4326').dissolve('route').reset_index()

print(f'\ncurrent file  : {len(geo_current):3d} routes')
print(f'2015 file     : {len(geo_2015):3d} routes')
print(f'  in both     : {len(set(geo_current.route) & set(geo_2015.route)):3d}')
print(f'  2015 only   : {len(set(geo_2015.route) - set(geo_current.route)):3d}')
print(f'  current only: {len(set(geo_current.route) - set(geo_2015.route)):3d}')

## 1. Data quality checks and cross checks

Nothing is dropped or filtered in this section — it only counts. Every check prints its
result even when the result is zero, so an absent problem is visible rather than assumed.

### 1.a) Missing data, NaNs, duplicates

In [ ]:
# ---------------------------------------------------------------------------
# INTEGRITY REPORT.  What weird stuff is in the data?
# ---------------------------------------------------------------------------
# `d` is the workhorse frame for the whole notebook: one row per route-day, starting as
# df_raw with dates parsed and rides coerced to numeric. It picks up derived columns as it
# goes -- week and dow in 1.c, name and corridor in 2, era in 4.c -- and is written out
# whole to data/derived/daily.csv in 6. Nothing is ever dropped from it.
d = df_raw.copy()
d['date']  = pd.to_datetime(d.date,  errors='coerce')
d['rides'] = pd.to_numeric(d.rides, errors='coerce')

for label, n in {
    'rows':                len(d),
    'unparseable dates':   int(d.date.isna().sum()),
    'non-numeric rides':   int(d.rides.isna().sum()),
    'rides < 0':           int((d.rides < 0).sum()),
    'rides == 0':          int((d.rides == 0).sum()),
    'null route':          int(d.route.isna().sum()),
    'null daytype':        int(d.daytype.isna().sum()),
}.items():
    print(f'{label:<28}: {n:>10,}')

print()
print(f'{"date range":<28}: {d.date.min().date()} .. {d.date.max().date()}')
print(f'{"distinct routes":<28}: {d.route.nunique():>10,}')
print(f'{"daytype values":<28}: {sorted(d.daytype.dropna().unique())}')

# keep=False flags every row of a duplicated pair, not just the second one.
is_duplicate = d.duplicated(subset=['route', 'date'], keep=False)
print(f'{"duplicate (route,date) rows":<28}: {int(is_duplicate.sum()):>10,}')
if is_duplicate.any():
    duplicate_groups = d.loc[is_duplicate].groupby(['route', 'date'])
    print(f'{"  affected route-days":<28}: {duplicate_groups.ngroups:>10,}')
    print(f'{"  identical rides in dup":<28}: '
          f'{int((duplicate_groups.rides.nunique() == 1).sum()):>10,}')
    display(d.loc[is_duplicate].sort_values(['route', 'date']).head(20))
    print(f'  (showing up to 20 of {int(is_duplicate.sum()):,} duplicate rows)')

In [ ]:
# Calendar coverage: is every day between the first and last date present?
calendar_days = pd.date_range(d.date.min(), d.date.max(), freq='D')
missing_days = calendar_days.difference(pd.Index(d.date.unique()))
print(f'{"calendar days in range":<28}: {len(calendar_days):>10,}')
print(f'{"days with no rows at all":<28}: {len(missing_days):>10,}')
if len(missing_days):
    print('  ', [str(x.date()) for x in missing_days[:20]],
          f'... (showing up to 20 of {len(missing_days)})')

# A day can be present but thin, so also count how many routes reported on each one.
routes_per_day = d.groupby('date').route.nunique()
print(f'\nroutes reporting per day:  min={routes_per_day.min()}   '
      f'median={routes_per_day.median():.0f}   max={routes_per_day.max()}')

### 1.b) Cross-check the daily file against the monthly file

`daytype` `W` excludes holidays, so a month's mean over `W` days should equal the published
`avg_weekday_rides`. Any systematic gap would mean we are reading the day types wrongly.

In [ ]:
# Our own per-route monthly mean over W days, to compare against the published figure.
our_weekday_mean = (d[d.daytype == 'W']
                      .groupby(['route', pd.Grouper(key='date', freq='MS')]).rides.mean()
                      .rename('ours').reset_index()
                      .rename(columns={'date': 'month_beginning'}))

# Inner join: only month-route pairs present in both files can be compared.
check = our_weekday_mean.merge(
    df_monthly[['route', 'month_beginning', 'avg_weekday_rides']],
    on=['route', 'month_beginning'], how='inner')
check['diff_pct'] = (check.ours - check.avg_weekday_rides) / check.avg_weekday_rides * 100

print(f'month-route pairs compared : {len(check):,}')
print(f'  exact to within 0.5%     : {int((check.diff_pct.abs() < 0.5).sum()):,}')
print(f'  median |difference|      : {check.diff_pct.abs().median():.3f}%')
print(f'  worst |difference|       : {check.diff_pct.abs().max():.2f}%')
print('\nlargest disagreements:')
print(check.reindex(check.diff_pct.abs().sort_values(ascending=False).index)
           .head(8)[['route', 'month_beginning', 'ours', 'avg_weekday_rides', 'diff_pct']]
           .to_string(index=False))

# Which routes appear in one file but not the other?
print(f'\nin daily but not monthly (no name available): '
      f'{sorted(set(d.route) - set(df_monthly.route))}')
print(f'in monthly but not daily: {sorted(set(df_monthly.route) - set(d.route))}')

### 1.c) Weeks, and endpoint issues

Weeks are **Mon–Sun**, labelled by the Monday that starts them, derived from `date` alone.
Each week carries `days` = how many distinct calendar days actually appear in the data, so a
short week at either end of the record is visible rather than silently reading as a dip.

In [ ]:
# Subtracting the weekday index from the date snaps every date back to its own Monday.
d['week'] = d.date - pd.to_timedelta(d.date.dt.weekday, unit='D')
d['dow']  = d.date.dt.dayofweek                 # 0=Mon .. 6=Sun, from the date itself
DOW = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

# One row per Mon-Sun week. `days` counts the calendar days that actually appear, so a
# short week at either end of the record shows up as short rather than as a dip.
df_weekly = (d.groupby('week')
               .agg(rides=('rides', 'sum'), days=('date', 'nunique'),
                    routes=('route', 'nunique'))
               .reset_index())
df_weekly['partial'] = df_weekly.days < 7

print(f'weeks spanned            : {len(df_weekly):,}')
print(f'partial weeks (<7 days)  : {int(df_weekly.partial.sum())}')
print()
print(df_weekly[df_weekly.partial].to_string(index=False))

## 2. Corridors: how do we treat routes that cover the same streets?

**Corridor**, starting definition: routes sharing the same numeric root — so `49`, `X49`, `49B`
belong to corridor `49`, and `J14` to corridor `14`. This is a first cut, not a final answer;
branch and express routes do not always follow the same street, so the families printed below
are meant to be audited one at a time before any of them are actually summed.

`R` routes are the 2013 Red Line South reconstruction shuttles, numbered for the **station**
served rather than the street run — `R95` ran the Dan Ryan from 95th to Garfield, while route
`95` is the 95th Street crosstown. They get no corridor, and are not dropped: their rides stay
in every total, including §5.

`check_r_routes.py` is the working for that — which corridors they would otherwise have joined,
and what keeping them in costs the day-of-week shape.

In [ ]:
def corridor(route):
    """Numeric root of a route id: X49 -> 49, J14 -> 14, 1001 -> 1001."""
    match = re.search(r'\d+', route)
    return match.group() if match else route

d['name'] = d.route.map(route_names)

# The R-prefixed routes are the 2013 Red Line South reconstruction shuttles (see above).
# Their number is the STATION they served, not the street they ran on, so the numeric-root
# rule files them under corridors they never touched -- four of them under Frequent Network
# corridors. They are given no corridor here. They are NOT dropped: their rides stay in d
# and in every system total. The routes matched are printed so that a future data refresh
# cannot silently change what this filter catches.
is_rail_replacement = d.route.str.match(r'^R\d', na=False)
d['corridor'] = d.route.map(corridor).where(~is_rail_replacement)

rail_replacement = (d[is_rail_replacement].groupby('route')
                      .agg(name=('name', 'first'), first=('date', 'min'),
                           last=('date', 'max'), days=('date', 'nunique'),
                           rides_per_day=('rides', 'mean')))
for col in ('first', 'last'):
    rail_replacement[col] = rail_replacement[col].dt.strftime('%Y-%m-%d')
print(f'rail-replacement routes given no corridor: {len(rail_replacement)}')
print(rail_replacement.round(0).to_string(), '\n')

# route_mean exists only to order each corridor's routes largest-first in the listing below.
route_mean = d.groupby('route').rides.mean()
families = (d.groupby('corridor').route.unique()
              .loc[lambda s: s.map(len) > 1]        # keep corridors holding >1 route
              .sort_index(key=lambda i: i.astype(int)))

print(f'corridors: {d.corridor.nunique()}   of which multi-route: {len(families)}')
print('Read this list critically -- some of these are one street, others are not.\n')
for corridor_id, corridor_routes in families.items():
    print(f'  corridor {corridor_id}')
    for route in sorted(corridor_routes, key=lambda x: -route_mean[x]):
        print(f'      {route:<6} {route_mean[route]:>8,.0f}/day   '
              f'{route_names.get(route, "(no name)")}')

## 3. Total ridership for the full dataset, 2001-2026

By week. The lower panel is the number of routes reporting that week. The system total is a sum
over a route set that changes over time, so the two have to be read together.

In [ ]:
df_weekly['era'] = df_weekly.week.dt.year.map(era)

fig, (ax_rides, ax_routes) = plt.subplots(
    2, 1, figsize=(11, 5.8), sharey=False, sharex=True,
    gridspec_kw={'height_ratios': [3, 1], 'hspace': 0.12})

# One coloured line segment per era. Eras are contiguous in time, so each slice is extended
# by one week past its last row to close the seam with the next.
for era_name in ERA_ORDER:
    era_rows = np.flatnonzero((df_weekly.era == era_name).to_numpy())
    era_slice = slice(era_rows[0], era_rows[-1] + 2)
    ax_rides.plot(df_weekly.week[era_slice], df_weekly.rides[era_slice],
                  lw=1.0, color=ERA_COLOR[era_name], label=era_name)

# Only draw the partial-week marker if there are any; the count is printed above either way.
# Drawn before the legend is built so that its label actually appears in the legend.
if df_weekly.partial.any():
    ax_rides.scatter(df_weekly.loc[df_weekly.partial, 'week'],
                     df_weekly.loc[df_weekly.partial, 'rides'],
                     s=30, color=PURPLE, zorder=3, label='partial week (<7 days of data)')

ax_rides.set_ylabel('rides per week')
ax_rides.yaxis.set_major_formatter(fmt_riders)
ax_rides.set_title('CTA bus ridership by week — entire dataset', loc='left', fontsize=11)
ax_rides.legend(frameon=False, loc='lower left', ncol=4)

# Lower panel: the route count behind the total, since the route set changes over time.
ax_routes.plot(df_weekly.week, df_weekly.routes, lw=1.0, color=INK)
ax_routes.set_ylabel('routes\nreporting')
ax_routes.set_xlabel('week (Monday)')
plt.show()

## 4. Explore ridership by route

### 4.a) The route network, mapped

The two geometry files read in §0.d, coloured by ridership. Each panel pairs a geometry vintage
with a ridership window that matches it: calendar 2014 for the 2015 file, the most recent year
in the data for the current file. One shared colour scale.

Routes not reporting in a panel's window are not drawn, and are listed. The scale is linear and
clipped at both ends; counts at the floor and the cap are printed.

The orientation classes built here are what §4.c splits the lifespan plot by, and the route
lengths are the denominator in §4.b.

In [ ]:
# How far the geometry gets us: which ridership routes can be drawn and measured at all.
routes_in_data = set(d.route.unique())
routes_with_geometry = set(geo_current.route) | set(geo_2015.route)
print(f'routes in the ridership data : {len(routes_in_data)}')
print(f'  with geometry in either file: {len(routes_in_data & routes_with_geometry)}')
print(f'  with no geometry at all     : {len(routes_in_data - routes_with_geometry)}')
print(f'geometry routes absent from the ridership data: '
      f'{sorted(routes_with_geometry - routes_in_data)}')

In [ ]:
# Orientation of each route, from its geometry. Every segment's bearing is folded to 0-90
# degrees from east and its LENGTH added to one of three bands, so a short downtown loop
# cannot outvote a long straight run. A route is called N-S / E-W / diagonal when one band
# holds a majority of its length; otherwise mixed. Nothing here touches the ridership data.
NS_MIN, EW_MAX, MAJORITY = 65.0, 25.0, 0.55

def orientation(geom):
    """Length-share N-S / E-W / diagonal, plus the length-weighted centroid, for one route."""
    seg_mids, seg_vectors, seg_lens = [], [], []
    for line in (geom.geoms if hasattr(geom, 'geoms') else [geom]):
        coords = np.asarray(line.coords)[:, :2]
        deltas = np.diff(coords, axis=0)                 # one vector per segment
        seg_mids.append((coords[:-1] + coords[1:]) / 2)
        seg_vectors.append(deltas)
        seg_lens.append(np.hypot(deltas[:, 0], deltas[:, 1]))
    mids = np.vstack(seg_mids)
    deltas = np.vstack(seg_vectors)
    lengths = np.concatenate(seg_lens)
    nonzero = lengths > 0                                # zero-length segments carry no bearing
    mids, deltas, lengths = mids[nonzero], deltas[nonzero], lengths[nonzero]

    bearings = np.abs(np.degrees(np.arctan2(deltas[:, 1], deltas[:, 0])))
    bearings = np.minimum(bearings, 180 - bearings)      # undirected: fold to 0-90 from east
    ns_share = lengths[bearings >= NS_MIN].sum() / lengths.sum()
    ew_share = lengths[bearings <= EW_MAX].sum() / lengths.sum()
    return pd.Series({'ns': ns_share, 'ew': ew_share,
                      'dg': max(0.0, 1 - ns_share - ew_share),
                      # centroid weighted by length, so it sits on the bulk of the route
                      'cx': np.average(mids[:, 0], weights=lengths),
                      'cy': np.average(mids[:, 1], weights=lengths),
                      'len_mi': lengths.sum() / 5280})   # CRS is in feet

# Current geometry preferred; the 2015 file only fills in routes the current file has dropped.
geo_all = pd.concat(
    [geo_current.assign(geom_src='current'),
     geo_2015[~geo_2015.route.isin(set(geo_current.route))].assign(geom_src='2015')],
    ignore_index=True)
geo_all = gpd.GeoDataFrame(geo_all, crs='EPSG:4326').to_crs(CRS)

# route_geo: one row per route with geometry, carrying everything measured off it --
# orientation shares, class, panel, centroid, length in miles, and which file it came from.
route_geo = pd.concat([geo_all[['route', 'geom_src']].reset_index(drop=True),
                       geo_all.geometry.apply(orientation).reset_index(drop=True)], axis=1)
top_band = route_geo[['ns', 'ew', 'dg']].idxmax(axis=1)    # which band holds the most length
top_share = route_geo[['ns', 'ew', 'dg']].max(axis=1)      # and how much of it
route_geo['orient'] = np.where(
    top_share >= MAJORITY,
    top_band.map({'ns': 'North-South', 'ew': 'East-West', 'dg': 'Diagonal'}),
    'Mixed / other')
route_geo['orient_share'] = top_share
# Three panels, so diagonal and mixed share the third. Chicago's diagonals are mixed by
# length -- Archer is 30/29/41 ns/ew/dg -- so few routes come out a diagonal majority.
route_geo['panel'] = route_geo.orient.map({'North-South': 'North-South',
                                           'East-West': 'East-West',
                                           'Diagonal': 'Diagonal / other',
                                           'Mixed / other': 'Diagonal / other'})
route_geo = route_geo.set_index('route')

print(f'orientation of {len(route_geo)} routes with geometry  '
      f'(N-S >= {NS_MIN:.0f} deg, E-W <= {EW_MAX:.0f} deg, majority >= {MAJORITY:.0%})')
print(route_geo.orient.value_counts().to_string())
print('\npanels:')
print(route_geo.panel.value_counts().to_string())
print('\ngeometry taken from: '
      + ', '.join(f'{k} {v}' for k, v in route_geo.geom_src.value_counts().items()))
print('\nroutes with a diagonal majority:')
print(route_geo[route_geo.orient == 'Diagonal'][['ns', 'ew', 'dg', 'len_mi']]
      .round(2).to_string())

In [ ]:
DATA_END = d.date.max()
MAP_WINDOWS = {                                  # geometry vintage -> matching ridership window
    '2015':    (pd.Timestamp('2014-01-01'), pd.Timestamp('2014-12-31')),
    'current': (DATA_END - pd.DateOffset(years=1) + pd.Timedelta(days=1), DATA_END),
}
MAP_TITLE = {'2015':    'Geometry 2015-02, ridership {w0:%Y-%m} to {w1:%Y-%m}',
             'current': 'Geometry 2025-01, ridership {w0:%Y-%m} to {w1:%Y-%m}'}
VMIN, VMAX = 500, 10_000        # linear, clipped at both ends; the counts are printed below

# Mean riders/day per route within each panel's own window.
map_mean = {}
for vintage, (win_start, win_end) in MAP_WINDOWS.items():
    window_rows = d[(d.date >= win_start) & (d.date <= win_end)]
    map_mean[vintage] = window_rows.groupby('route').rides.mean()
    print(f'{vintage:8s} {win_start.date()} to {win_end.date()}: '
          f'{len(window_rows):,} route-days, {window_rows.route.nunique()} routes reporting')

print(f'\ncolour scale {VMIN:,} to {VMAX:,} mean riders/day, shared by both panels:')
for vintage, means in map_mean.items():
    print(f'  {vintage:8s} {int((means > VMAX).sum()):3d} of {len(means)} routes at the cap '
          f'(max {means.max():,.0f}), {int((means < VMIN).sum()):3d} at the floor')

# plasma truncated at 0.9: its top end is pale yellow, which disappears against white and
# would make the busiest routes the least visible.
map_cmap = LinearSegmentedColormap.from_list(
    'plasma_trunc', plt.get_cmap('plasma')(np.linspace(0.0, 0.90, 256)))
map_norm = Normalize(vmin=VMIN, vmax=VMAX)
map_geo  = {'2015':    gpd.GeoDataFrame(geo_2015, crs='EPSG:4326').to_crs(CRS),
            'current': gpd.GeoDataFrame(geo_current, crs='EPSG:4326').to_crs(CRS)}

fig, axes = plt.subplots(1, 2, figsize=(13.5, 8.2))
for ax, vintage in zip(axes, ['2015', 'current']):
    panel_geo = map_geo[vintage]
    panel_mean = map_mean[vintage]
    win_start, win_end = MAP_WINDOWS[vintage]
    # Build one flat list of line segments so the whole panel draws as a single collection.
    segments, colors, widths, skipped = [], [], [], []
    for _, row in panel_geo.iterrows():
        mean_rides = panel_mean.get(row.route, np.nan)
        if not np.isfinite(mean_rides) or mean_rides <= 0:
            skipped.append(row.route)             # not reporting in this window: not drawn
            continue
        scaled = float(np.clip(map_norm(mean_rides), 0, 1))
        for line in (row.geometry.geoms if hasattr(row.geometry, 'geoms') else [row.geometry]):
            segments.append(np.asarray(line.coords)[:, :2])
            colors.append(map_cmap(scaled))
            widths.append(0.7 + 2.6 * scaled)     # width echoes colour, so it survives in mono
    ax.add_collection(LineCollection(segments, colors=colors, linewidths=widths,
                                     capstyle='round'))
    ax.autoscale_view(); ax.set_aspect('equal'); ax.set_axis_off()
    ax.set_title(MAP_TITLE[vintage].format(w0=win_start, w1=win_end)
                 + f'\n{len(panel_geo) - len(skipped)} routes drawn'
                 + (f', {len(skipped)} not reporting' if skipped else ''),
                 loc='left', fontsize=9)
    print(f'\n{vintage:8s} drawn {len(panel_geo) - len(skipped)} of {len(panel_geo)}; '
          f'{len(skipped)} dropped for no ridership in the window'
          + (f': {", ".join(sorted(skipped))}' if skipped else ''))

# Same extent in both panels, so the networks are directly comparable.
x_limits = [min(a.get_xlim()[0] for a in axes), max(a.get_xlim()[1] for a in axes)]
y_limits = [min(a.get_ylim()[0] for a in axes), max(a.get_ylim()[1] for a in axes)]
for ax in axes:
    ax.set_xlim(x_limits); ax.set_ylim(y_limits)

colorbar = fig.colorbar(ScalarMappable(norm=map_norm, cmap=map_cmap), ax=axes,
                        fraction=0.035, pad=0.02, aspect=34)
colorbar.set_label(f'mean riders/day (clipped to {VMIN:,}-{VMAX:,})')
# fmt_riders floors anything under 1k to "0k", which this scale needs to show
colorbar.ax.yaxis.set_major_formatter(
    FuncFormatter(lambda v, _: f'{v*1e-3:g}k' if v >= 1000 else f'{v:g}'))
fig.suptitle('CTA bus network coloured by ridership — two snapshots',
             x=0.09, ha='left', fontsize=11)
plt.show()

### 4.b) Which routes have the highest ridership

Every route in the dataset by week, one line each. The 20 Frequent Network routes are highlighted.

Log scale, because route sizes span orders of magnitude. Nothing is excluded.

> **Note.** Only the 20 route IDs as CTA labels them are highlighted. Their express / branch
> variants (`X49`, `53A`, …) are drawn in grey with everything else, because the corridor
> question is still open.

In [ ]:
freq_in_data = [r for r in FREQ if r in routes_in_data]
print(f'Frequent Network routes (CTA labelling) : {len(FREQ)}')
print(f'  found in the data                     : {len(freq_in_data)}')
print(f'  NOT found in the data                 : '
      f'{[r for r in FREQ if r not in routes_in_data]}')

# Weeks down, routes across. NaN where a route did not report that week -- plotting the
# columns straight from this leaves real gaps rather than joining across them.
route_week_pivot = d.pivot_table(index='week', columns='route', values='rides', aggfunc='sum')
print(f'\nroute x week matrix: {route_week_pivot.shape[0]:,} weeks x '
      f'{route_week_pivot.shape[1]:,} routes')
print(f'empty cells (route not reporting that week): '
      f'{int(route_week_pivot.isna().sum().sum()):,}  of {route_week_pivot.size:,}')

In [ ]:
other_routes = [r for r in route_week_pivot.columns if r not in freq_in_data]

fig, ax = plt.subplots(figsize=(11, 5.6))
# Grey first, orange second, so the Frequent Network lines sit on top of the pack.
ax.plot(route_week_pivot.index, route_week_pivot[other_routes],
        lw=0.5, color=GRAY, alpha=0.55)
ax.plot(route_week_pivot.index, route_week_pivot[freq_in_data],
        lw=0.9, color=ORANGE, alpha=0.85)
ax.set_yscale('log')
ax.set_ylabel('rides per week (log scale)')
ax.set_xlabel('week (Monday)')
ax.set_title('Every bus route by week — Frequent Network routes highlighted',
             loc='left', fontsize=11)
# Proxy handles: 188 plotted lines would otherwise mean 188 legend entries.
ax.legend(handles=[Line2D([], [], color=ORANGE, lw=1.6,
                          label=f'Frequent Network ({len(freq_in_data)})'),
                   Line2D([], [], color=GRAY, lw=1.6,
                          label=f'all other routes ({len(other_routes)})')],
          frameon=False, loc='lower left')
plt.show()

#### Route inventory and ridership statistics

Unit throughout is **riders per day** — the raw records are daily totals, so a route's mean is
its mean over the days it reported. Weekdays, Saturdays and Sundays are pooled here, so a
route's mean reflects its weekday/weekend mix as well as its size.

`status` marks routes that stopped reporting more than 30 days before the end of the data —
these are routes that were cut or renumbered. Nothing is filtered; all 188 routes are listed.

In [ ]:
LAST_DATE = d.date.max()
by_route = d.groupby('route')          # reused for every statistic below

# route_inventory: one row per route, the summary table the rest of section 4 works from.
route_inventory = pd.DataFrame({
    'name':     by_route.name.first(),
    'corridor': by_route.corridor.first(),
    'first':    by_route.date.min(),
    'last':     by_route.date.max(),
    'days':     by_route.date.nunique(),
    'mean':     by_route.rides.mean(),
    'median':   by_route.rides.median(),
    'std':      by_route.rides.std(),
    'min':      by_route.rides.min(),
    'max':      by_route.rides.max(),
})
# idxmin/idxmax give the row label of the extreme day, which is then looked up for its date.
route_inventory['min_date'] = d.loc[by_route.rides.idxmin(),
                                    ['route', 'date']].set_index('route').date
route_inventory['max_date'] = d.loc[by_route.rides.idxmax(),
                                    ['route', 'date']].set_index('route').date
# A route still reporting within 30 days of the end of the data counts as active.
route_inventory['status'] = np.where(
    route_inventory['last'] >= LAST_DATE - pd.Timedelta(days=30),
    'active', 'ended ' + route_inventory['last'].dt.strftime('%Y-%m'))

route_inventory = route_inventory.sort_values('mean', ascending=False)
for col in ('first', 'last', 'min_date', 'max_date'):
    route_inventory[col] = route_inventory[col].dt.strftime('%Y-%m-%d')

print(f'routes: {len(route_inventory)}   '
      f'active: {int((route_inventory.status == "active").sum())}   '
      f'ended: {int((route_inventory.status != "active").sum())}')
with pd.option_context('display.max_rows', None, 'display.width', 200):
    display(route_inventory[['name', 'corridor', 'first', 'last', 'days', 'status', 'mean',
                             'median', 'std', 'min', 'min_date', 'max', 'max_date']].round(1))

#### Route size against route size per mile

The same means as a plot, but against route length: **mean riders/day** on the x axis, **mean
riders/day/mile** on the y. Raw boardings favour long routes, so the y axis asks how heavily
used a route is for the amount of street it covers. Both axes linear. All 20 Frequent Network
routes are labelled.

Length comes from `route_geo` (§4.a), and has three cases, all printed by the cell:

- **own** — the route appears in one of the two geometry files.
- **parent** — an `X` express with no geometry of its own, given its parent route's length
  (`X54` → `54`). **These need checking against CTA maps** — an express does not necessarily
  run the full length of its parent, and all seven last reported 2005–2010, so a current-file
  length is being applied to a mean from fifteen years earlier.
- **none** — no length anywhere. Drawn on the x axis with no y value.

The 2013 Red Line South `R` shuttles are excluded outright as a one-off rail replacement.

Markers carry the caveats: **open** = length from the deprecated 2015 file, **✕** = route
stopped reporting.

In [ ]:
# df_route_size: one row per route with the two size measures and where its length came from.
df_route_size = route_inventory[['name', 'mean', 'status']].copy()
summed_mean = route_inventory['mean'].sum()          # denominator for every share printed

# --- 1. Drop the 2013 Red Line South shuttles -------------------------------------------
is_r_shuttle = df_route_size.index.str.match(r'^R\d')
r_shuttles = df_route_size[is_r_shuttle]
df_route_size = df_route_size[~is_r_shuttle]
print(f'EXCLUDED, 2013 Red Line South shuttles ({len(r_shuttles)}): '
      f'{", ".join(sorted(r_shuttles.index))}')
print(f'  {r_shuttles["mean"].sum() / summed_mean:.1%} of summed mean riders/day\n')

# --- 2. Own length, for routes present in either geometry file ---------------------------
length = route_geo['len_mi'].reindex(df_route_size.index)
length_src = route_geo['geom_src'].reindex(df_route_size.index)
length_from = pd.Series(np.where(length.notna(), 'own', 'none'), index=df_route_size.index)

# --- 3. X expresses with no geometry inherit their parent route's length ------------------
needs_parent = df_route_size.index[length.isna() & df_route_size.index.str.match(r'^X\d')]
inherited = []
for route in needs_parent:
    parent = corridor(route)                          # X54 -> 54, same helper as section 2
    if parent not in route_geo.index:
        continue
    length[route] = route_geo.loc[parent, 'len_mi']
    length_src[route] = route_geo.loc[parent, 'geom_src']
    length_from[route] = 'parent'
    inherited.append({'express': route, 'parent': parent,
                      'parent name': route_names.get(parent, '(no name)'),
                      'len_mi': route_geo.loc[parent, 'len_mi'],
                      'express last ran': route_inventory.loc[route, 'last']})

print(f'LENGTH INHERITED FROM PARENT ROUTE ({len(inherited)} of {len(needs_parent)} X routes)'
      ' -- confirm each against a CTA map:')
print(pd.DataFrame(inherited).round(1).to_string(index=False))
no_parent = [r for r in needs_parent if corridor(r) not in route_geo.index]
print(f'  X routes whose parent also has no geometry: {no_parent} -> drawn on the axis\n')

df_route_size['len_mi'] = length
df_route_size['len_src'] = length_src
df_route_size['len_from'] = length_from
df_route_size['riders_per_mile'] = df_route_size['mean'] / df_route_size['len_mi']

# --- 4. What is left ----------------------------------------------------------------------
has_length = df_route_size.len_mi.notna()
plotted, on_axis = df_route_size[has_length], df_route_size[~has_length]
print(f'plotted with a length : {len(plotted):3d}  '
      f'({plotted["mean"].sum() / summed_mean:.1%} of summed mean riders/day)')
print(f'  length from own geometry : {int((plotted.len_from == "own").sum()):3d}')
print(f'  length from parent route : {int((plotted.len_from == "parent").sum()):3d}')
print(f'  length from the 2015 file: {int((plotted.len_src == "2015").sum()):3d}  '
      f'(drawn as open markers; the ridership mean spans the whole record)')
print(f'drawn on the x axis, no length : {len(on_axis):3d}  '
      f'({on_axis["mean"].sum() / summed_mean:.1%} of summed mean riders/day)')
print(f'  {", ".join(sorted(on_axis.index))}')

# --- 5. Plot -------------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(11, 7.2))
is_frequent = plotted.index.isin(FREQ_SET)
is_active = (plotted.status == 'active').to_numpy()
from_2015 = (plotted.len_src == '2015').to_numpy()

# Four marker combinations per colour: circle/cross for still-running/stopped, filled/open
# for a current/2015 length. Grey drawn first so the Frequent Network sits on top.
for freq_group, color, marker_size, layer in [(False, GRAY, 30, 2), (True, ORANGE, 55, 4)]:
    for active, marker in [(True, 'o'), (False, 'X')]:
        for old_geometry in (False, True):
            sel = ((is_frequent == freq_group) & (is_active == active)
                   & (from_2015 == old_geometry))
            if not sel.any():
                continue
            ax.scatter(plotted['mean'][sel], plotted.riders_per_mile[sel],
                       marker=marker, s=marker_size,
                       facecolors='none' if old_geometry else color,
                       edgecolors=color, linewidths=0.9, zorder=layer)

# Routes with no length at all: a tick on the axis, at their riders/day but with no y value.
ax.scatter(on_axis['mean'], np.zeros(len(on_axis)), marker='|', s=90,
           color=INK, linewidths=0.8, alpha=0.55, zorder=2)

ax.set_xlim(0, plotted['mean'].max() * 1.08)
ax.set_ylim(-plotted.riders_per_mile.max() * 0.035, plotted.riders_per_mile.max() * 1.10)
ax.set_xlabel('mean riders/day')
ax.set_ylabel('mean riders/day/mile')
ax.xaxis.set_major_formatter(fmt_riders)
ax.set_title('Route size against route size per mile — whole dataset', loc='left', fontsize=11)


def spread_labels(positions, gap):
    """Nudge label positions apart until none are within `gap`, preserving their order."""
    positions = positions.astype(float).copy()
    order = np.argsort(positions)
    for _ in range(200):
        moved = False
        for lower, upper in zip(order[:-1], order[1:]):
            overlap = gap - (positions[upper] - positions[lower])
            if overlap > 1e-9:                       # push the pair apart, half each way
                positions[lower] -= overlap / 2
                positions[upper] += overlap / 2
                moved = True
        if not moved:
            break
    return positions


# Label every Frequent Network route, with a leader line back to its dot. Positions are
# worked out in axes fractions (0-1) so the minimum gap is a real distance on the page.
freq_points = plotted[is_frequent]
x_lo, x_hi = ax.get_xlim()
y_lo, y_hi = ax.get_ylim()
point_fx = ((freq_points['mean'] - x_lo) / (x_hi - x_lo)).to_numpy()
point_fy = ((freq_points.riders_per_mile - y_lo) / (y_hi - y_lo)).to_numpy()
label_text = [f'{r} {freq_points.loc[r, "name"]}' for r in freq_points.index]

# Labels sit to the right of their dot, except near the right edge where they would run off.
put_left = point_fx > 0.66
label_fy = point_fy.copy()
for side in (~put_left, put_left):                   # de-collide each column on its own
    if side.any():
        label_fy[side] = spread_labels(point_fy[side], 0.033)
label_fy = np.clip(label_fy, 0.012, 0.988)

for i, route in enumerate(freq_points.index):
    left = put_left[i]
    ax.annotate(label_text[i],
                xy=(freq_points['mean'].iloc[i], freq_points.riders_per_mile.iloc[i]),
                xycoords='data',
                xytext=(point_fx[i] + (-0.028 if left else 0.028), label_fy[i]),
                textcoords='axes fraction',
                ha='right' if left else 'left', va='center', fontsize=6.5, color=INK,
                arrowprops=dict(arrowstyle='-', lw=0.5, color=ORANGE,
                                shrinkA=1, shrinkB=2, alpha=0.85))

# Legend in the lower right: high riders/day but low riders/day/mile is an empty corner.
ax.legend(handles=[
    Line2D([], [], ls='', marker='o', color=ORANGE, ms=6,
           label=f'Frequent Network ({int(is_frequent.sum())})'),
    Line2D([], [], ls='', marker='o', color=GRAY, ms=5,
           label=f'other routes ({int((~is_frequent).sum())})'),
    Line2D([], [], ls='', marker='o', mfc='none', mec=INK, ms=6,
           label=f'open = length from the 2015 file ({int(from_2015.sum())})'),
    Line2D([], [], ls='', marker='X', mfc='none', mec=INK, ms=6,
           label=f'✕ = stopped reporting ({int((~is_active).sum())})'),
    Line2D([], [], ls='', marker='|', color=INK, ms=8,
           label=f'on the axis = no length available ({len(on_axis)})'),
], frameon=False, loc='lower right', fontsize=7.5, handletextpad=0.6)
plt.show()

cum_share = route_inventory['mean'].cumsum() / summed_mean   # already sorted largest first
print(f'top 20 routes carry {cum_share.iloc[19]:.0%} of mean daily boardings; '
      f'top 50 carry {cum_share.iloc[49]:.0%}')
print('\nhighest riders/day/mile:')
print(plotted.nlargest(10, 'riders_per_mile')[['name', 'mean', 'len_mi', 'riders_per_mile']]
             .round(0).to_string())

### 4.c) Route change over time

Same statistics split by era. A route absent from an era simply did not run then — those cells
are blank rather than zero.

2020 is kept as its own column rather than pooled with the recovery years, which would hide how
different it was. The bands themselves are a reading aid, not an analytical grouping — see §0.a.

The lifespan plot below splits routes into three panels by the orientation classes built in
§4.a, sorted geographically within each: N-S west to east, E-W and the third panel north to
south. Chicago's diagonals are mixed by length, so they sit in the third panel with the
circulators and the L-shaped routes.

In [ ]:
d['era'] = d.date.dt.year.map(era)

# Routes down, (statistic, era) across. unstack turns the era level into columns; reindex
# puts them back in chronological order, since groupby sorts them alphabetically.
per_era = (d.groupby(['route', 'era']).rides
             .agg(['mean', 'median', 'std', 'size'])
             .rename(columns={'size': 'days'})
             .unstack('era')
             .reindex(columns=ERA_ORDER, level=1))

per_era = per_era.reindex(route_inventory.index)     # keep the sort by overall mean
last_era = ERA_ORDER[-1]
print(f'{last_era} is a partial era: {d[d.era == last_era].date.max().date()} is the last '
      f'date in the data.')
with pd.option_context('display.max_rows', None, 'display.width', 200):
    display(per_era.round(1))

In [ ]:
# When each route ran, split by orientation (route_geo, §4.a) and sorted geographically
# within each panel. Routes with no geometry in either file cannot be placed, so they are
# listed below rather than assigned to a panel on a guess.
route_spans = (d.groupby('route').date.agg(['min', 'max'])
                 .join(route_inventory[['status', 'mean', 'name']])
                 .join(route_geo[['panel', 'cx', 'cy', 'geom_src']]))
no_geometry = route_spans[route_spans.panel.isna()]
route_spans = route_spans[route_spans.panel.notna()]
print(f'routes plotted: {len(route_spans)}    not plotted, no geometry: {len(no_geometry)} '
      f'({no_geometry["mean"].sum() / route_inventory["mean"].sum():.1%} of summed mean '
      f'riders/day)')

PANELS = ['North-South', 'East-West', 'Diagonal / other']
PANEL_SORT = {'North-South':      ('cx', True,  'west to east'),   # column, ascending, label
              'East-West':        ('cy', False, 'north to south'),
              'Diagonal / other': ('cy', False, 'north to south')}

panel_groups = {name: route_spans[route_spans.panel == name]
                            .sort_values(PANEL_SORT[name][0], ascending=PANEL_SORT[name][1])
                for name in PANELS}
n_rows = max(len(group) for group in panel_groups.values())
fig_height = 0.145 * n_rows + 1.4                # ~0.145in per route, plus room for the title

fig, axes = plt.subplots(1, 3, figsize=(16, fig_height), sharex=True)
fig.subplots_adjust(top=1 - 0.62 / fig_height, bottom=0.34 / fig_height)
for ax, panel_name in zip(axes, PANELS):
    panel_routes = panel_groups[panel_name]
    y_pos = np.arange(len(panel_routes))
    has_ended = (panel_routes.status != 'active').values
    ax.hlines(y_pos[~has_ended], panel_routes['min'][~has_ended],
              panel_routes['max'][~has_ended], color=BLUE, lw=2.0)
    ax.hlines(y_pos[has_ended], panel_routes['min'][has_ended],
              panel_routes['max'][has_ended], color=ORANGE, lw=2.0)
    ax.scatter(panel_routes['max'][has_ended], y_pos[has_ended], s=10, color=ORANGE, zorder=3)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(panel_routes.index, fontsize=5.6)
    # Frequent Network routes picked out in the tick labels themselves.
    for tick, route in zip(ax.get_yticklabels(), panel_routes.index):
        tick.set_fontweight('bold' if route in FREQ_SET else 'normal')
        tick.set_color(INK if route in FREQ_SET else '#8A8A8A')
    ax.set_ylim(-1, n_rows)        # same row pitch in every panel; the short ones end early
    ax.invert_yaxis()
    ax.tick_params(axis='y', length=0, pad=1.5)
    ax.grid(axis='y', visible=False)
    ax.set_xlabel('date')
    ax.set_title(f'{panel_name}  ({len(panel_routes)}, {PANEL_SORT[panel_name][2]})',
                 loc='left', fontsize=9.5)

fig.legend(handles=[Line2D([], [], color=BLUE,   lw=2.2, label='active'),
                    Line2D([], [], color=ORANGE, lw=2.2, label='stopped reporting'),
                    Line2D([], [], color='none', label='bold = Frequent Network')],
           frameon=False, ncol=3, loc='lower right',
           bbox_to_anchor=(0.995, 1 - 0.52 / fig_height), fontsize=8)
fig.suptitle('Route lifespans, by orientation and geographic position',
             x=0.008, y=1 - 0.16 / fig_height, ha='left', va='top', fontsize=11)
plt.show()


def span_table(frame, n=15):
    """Largest-first span table, for the routes that ended and the ones with no geometry."""
    return (frame.nlargest(n, 'mean')[['name', 'min', 'max', 'mean']]
                 .assign(min=lambda t: t['min'].dt.strftime('%Y-%m'),
                         max=lambda t: t['max'].dt.strftime('%Y-%m'))
                 .rename(columns={'min': 'first', 'max': 'last', 'mean': 'riders/day'})
                 .round(0).to_string())

ended_routes = route_spans[route_spans.status != 'active']
print(f'plotted routes that stopped reporting ({len(ended_routes)}), largest first:')
print(span_table(ended_routes))
print(f'(showing up to 15 of {len(ended_routes)})')

print(f'\nnot plotted, no geometry in either file ({len(no_geometry)}), largest first:')
print(span_table(no_geometry))
print(f'(showing up to 15 of {len(no_geometry)})')

### 4.d) Route recovery since the pandemic

Mean riders/day in 2025 against each route's own pre-pandemic level, log-log.

Two panels because the baseline is a choice: the whole pre-pandemic record (2001–2019), and the
level immediately before the pandemic (2018–2019). Ridership fell 2–3% a year from 2015, so the
two differ. 2025 is itself a rollout year.

In [ ]:
# Recovery: each route's mean riders/day in a recent year against its own pre-pandemic
# baseline. Both the recent year and the baselines are named here rather than taken from
# the era bands, so the choice is explicit and easy to change.
RECENT_YEAR = 2025                                  # last complete calendar year
BASELINES   = {'2001-2019': (2001, 2019),           # the whole pre-pandemic record
               '2018-2019': (2018, 2019)}           # the level just before the pandemic

year = d.date.dt.year
recent_mean = d[year == RECENT_YEAR].groupby('route').rides.mean()

# One frame per baseline. dropna keeps only routes that ran in both windows -- a route that
# started after 2019 or ended before 2025 has no ratio to report.
recoveries = {}
for label, (lo, hi) in BASELINES.items():
    baseline_mean = d[(year >= lo) & (year <= hi)].groupby('route').rides.mean()
    recovery = pd.DataFrame({'pre': baseline_mean, 'now': recent_mean,
                             'name': route_inventory.name}).dropna(subset=['pre', 'now'])
    recovery['ratio'] = recovery.now / recovery.pre
    recoveries[label] = recovery

# Shared limits so the two panels can be read against each other.
all_values = np.concatenate([r[['pre', 'now']].to_numpy().ravel() for r in recoveries.values()])
axis_limits = [all_values.min() * 0.7, all_values.max() * 1.4]

fig, axes = plt.subplots(1, 2, figsize=(12.4, 6.0), sharex=True, sharey=True)
for ax, (label, recovery) in zip(axes, recoveries.items()):
    is_frequent = recovery.index.isin(FREQ_SET)
    ax.plot(axis_limits, axis_limits, color=INK, lw=0.9, ls=':', label='no change')
    ax.plot(axis_limits, [v * 0.5 for v in axis_limits], color=GRAY, lw=0.9, ls='--',
            label='half of baseline')
    ax.scatter(recovery.pre[~is_frequent], recovery.now[~is_frequent], s=16, color=GRAY)
    ax.scatter(recovery.pre[is_frequent], recovery.now[is_frequent], s=26, color=ORANGE,
               zorder=3, label='Frequent Network')
    # Label only the extremes at each end of the ratio.
    for route in recovery.ratio.nlargest(4).index.union(recovery.ratio.nsmallest(4).index):
        ax.annotate(route, (recovery.loc[route, 'pre'], recovery.loc[route, 'now']),
                    xytext=(5, 3), textcoords='offset points', fontsize=7)
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlim(axis_limits); ax.set_ylim(axis_limits)
    ax.set_xlabel(f'mean riders/day, {label} baseline')
    ax.set_title(f'baseline {label}   (median ratio {recovery.ratio.median():.2f})',
                 loc='left', fontsize=10)

axes[0].set_ylabel(f'mean riders/day, {RECENT_YEAR}')
axes[0].legend(frameon=False, loc='upper left')
fig.suptitle(f'Recovery by route — {RECENT_YEAR} against two pre-pandemic baselines',
             x=0.125, ha='left', fontsize=11)
plt.show()

for label, recovery in recoveries.items():
    print(f'baseline {label}  ({len(recovery)} routes with both)')
    print(f'  routes at or above baseline : {int((recovery.ratio > 1).sum())}')
    print(f'  median recovery ratio       : {recovery.ratio.median():.2f}')

recovery = recoveries['2018-2019']
print('\nstrongest and weakest against the 2018-2019 baseline:')
print(pd.concat([recovery.nlargest(5, 'ratio'), recovery.nsmallest(5, 'ratio')])
        [['name', 'pre', 'now', 'ratio']].round(2).to_string())

## 5. Explore ridership by day of the week

Three views, because "by day of week" means different things at different scales:

- **(a)** absolute mean rides per day, one line per year — dominated by the level change.
- **(b)** the same lines divided by each year's own mean, which is the test of whether the
  *shape* is stable year to year.
- **(c)** the spread within a single recent year, so the shape in (b) can be read against how
  much an individual day actually moves.

In [ ]:
# System-wide total for each calendar day, i.e. every route summed.
system_daily = d.groupby('date', as_index=False).rides.sum()
system_daily['dow']  = system_daily.date.dt.dayofweek
system_daily['year'] = system_daily.date.dt.year
system_daily['era']  = system_daily.year.map(era)

dow_by_year = system_daily.pivot_table(index='year', columns='dow', values='rides',
                                       aggfunc='mean')
dow_shape = dow_by_year.div(dow_by_year.mean(axis=1), axis=0)   # each year by its own mean
DETAIL_YEAR = 2025                                             # last complete calendar year

fig, axes = plt.subplots(1, 3, figsize=(13, 4.0))

# (a) and (b) are the same 26 lines, one per year, coloured by era.
for year in dow_by_year.index:
    color = ERA_COLOR[era(year)]
    axes[0].plot(range(7), dow_by_year.loc[year], color=color, lw=1.2)
    axes[1].plot(range(7), dow_shape.loc[year], color=color, lw=1.2)

axes[0].set_title('(a) mean rides per day', loc='left', fontsize=10)
axes[0].yaxis.set_major_formatter(fmt_riders)
axes[1].set_title("(b) shape: divided by each year's mean", loc='left', fontsize=10)
axes[1].axhline(1, color=INK, lw=0.8, ls=':')
axes[1].legend(handles=[Line2D([], [], color=ERA_COLOR[n], lw=1.6, label=n) for n in ERA_ORDER],
               frameon=False, fontsize=8, loc='lower left')

# (c) every individual day of one year, jittered sideways so the points do not stack up.
detail_days = system_daily[system_daily.year == DETAIL_YEAR]
for i in range(7):
    rides = detail_days.loc[detail_days.dow == i, 'rides']
    axes[2].scatter(i + np.random.uniform(-.16, .16, len(rides)), rides,
                    s=7, color=BLUE, alpha=0.35, linewidths=0)
    p10, p50, p90 = np.percentile(rides, [10, 50, 90])
    axes[2].plot([i - .3, i + .3], [p50, p50], color=ORANGE, lw=2, zorder=3)
    axes[2].plot([i, i], [p10, p90], color=ORANGE, lw=1, zorder=3)
axes[2].set_title(f'(c) every day in {DETAIL_YEAR}: median, 10-90th pct', loc='left',
                  fontsize=10)
axes[2].yaxis.set_major_formatter(fmt_riders)

for ax in axes:
    ax.set_xticks(range(7)); ax.set_xticklabels(DOW)
plt.show()

**On the statistic.** Max-minus-min across years is a poor summary — it is decided by whichever
single year is most extreme, which here is 2020. Standard deviation and inter-quartile range
across years are reported instead, and separately by era, so a genuinely stable shape can be
told apart from one held together by averaging.

In [ ]:
def dow_spread(shape_frame):
    """Across-year spread of the normalised day-of-week shape, one column per weekday."""
    return pd.DataFrame({
        'std':     shape_frame.std(),
        'IQR':     shape_frame.quantile(.75) - shape_frame.quantile(.25),
        'max-min': shape_frame.max() - shape_frame.min(),
    }).rename(index=lambda i: DOW[i]).T

print('all years (2001-2026)')
print(dow_spread(dow_shape).round(3).to_string())

# Same statistics within each era, so a stable shape can be told from an averaged one.
for era_name in ERA_ORDER:
    era_years = [y for y in dow_shape.index if era(y) == era_name]
    print(f'\n{era_name}  (n={len(era_years)} years)')
    print('  single year - no across-year spread defined' if len(era_years) < 2
          else dow_spread(dow_shape.loc[era_years]).round(3).to_string())

## 6. Save the cleaned data for the companion notebooks

`d` picks up its derived columns across sections 1–4, so it is written once, here, rather than
in pieces. `holidays.ipynb` and `seasonality.ipynb` read these files instead of re-deriving
them. `data/` is gitignored, so nothing here is committed.

In [ ]:
DERIVED = Path('data/derived')
DERIVED.mkdir(parents=True, exist_ok=True)

daily_path = DERIVED / 'daily.csv'
d.to_csv(daily_path, index=False)
print(f'{str(daily_path):<32} {len(d):>10,} rows   '
      f'{daily_path.stat().st_size / 1e6:.0f} MB')
print(f'  columns: {list(d.columns)}')

inventory_path = DERIVED / 'route_inventory.csv'
route_inventory.to_csv(inventory_path)
print(f'{str(inventory_path):<32} {len(route_inventory):>10,} rows')
print(f'  columns: {list(route_inventory.columns)}')

## Where this leaves us

### Established

- No duplicates, no gaps, no missing days; day-type handling reproduces the published monthly
  averages to 0.001% (§1.b).
- Day-of-week shape varies ~1-2% across years within a band; 2021-2022 is looser, up to 4.6% (§5).
- `R` routes given no corridor, kept in all ridership totals (§2).
- Recovery depends on the baseline: median route at 0.66 of 2001-2019, 0.81 of 2018-2019 (§4.d).
- 149 of 188 routes have geometry in one of the two portal files; the other 39 are 8.5% of
  summed mean riders/day (§4.a).

### Decisions still open

1. How express / branch routes fold into corridors (`X49`, `53A`, …).
2. Which pre-pandemic baseline the before/after uses.

### Continued elsewhere

- `holidays.ipynb`, `seasonality.ipynb`